# Surrogate Gradient Methods for Spiking Neural Networks

Learn how to train SNNs using surrogate gradients with JAX custom VJP.

## Table of Contents
1. [The Gradient Problem in SNNs](#1-the-gradient-problem-in-snns)
2. [Gaussian Pseudo-Derivative](#2-gaussian-pseudo-derivative)
3. [Implementation with jax.custom_vjp](#3-implementation-with-jaxcustom_vjp)
4. [Comparing Surrogate Methods](#4-comparing-surrogate-methods)
5. [Gradient Flow Verification](#5-gradient-flow-verification)

---

In [ ]:
# Setup
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# Enable high-DPI display for plots
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. The Gradient Problem in SNNs

Spiking Neural Networks (SNNs) generate discrete spike events using the **Heaviside step function**:

$$
z = H(v - v_{\text{th}}) = \begin{cases} 1 & \text{if } v > v_{\text{th}} \\ 0 & \text{otherwise} \end{cases}
$$

The problem: The Heaviside function has **zero gradient almost everywhere**, and an undefined gradient at the threshold. This makes standard backpropagation impossible.

$$
\frac{\partial H}{\partial v} = \begin{cases} 0 & v \neq 0 \\ \text{undefined} & v = 0 \end{cases}
$$

In [ ]:
# Visualize the Heaviside function and its derivative
v = np.linspace(-2, 2, 500)

# Heaviside (forward)
heaviside = np.where(v > 0, 1.0, 0.0)

# True derivative (zero everywhere, undefined at 0)
true_grad = np.zeros_like(v)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Forward pass
axes[0].plot(v, heaviside, 'b-', linewidth=2)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Scaled voltage (v - v_th)')
axes[0].set_ylabel('Spike output z')
axes[0].set_title('Forward: Heaviside Step Function')
axes[0].set_ylim([-0.1, 1.1])
axes[0].grid(True, alpha=0.3)

# True gradient
axes[1].plot(v, true_grad, 'r-', linewidth=2)
axes[1].scatter([0], [0], s=100, c='red', marker='x', zorder=5, label='Undefined')
axes[1].set_xlabel('Scaled voltage')
axes[1].set_ylabel('dz/dv')
axes[1].set_title('True Gradient: Zero Everywhere!')
axes[1].set_ylim([-0.1, 1.0])
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Problem: Gradients are zero, so weights cannot be updated via backpropagation!")

## 2. Gaussian Pseudo-Derivative

**Solution**: Use a **surrogate gradient** in the backward pass while keeping the Heaviside in the forward pass.

The V1 model uses a **Gaussian pseudo-derivative**:

$$
\frac{\partial z}{\partial v} \approx \exp\left(-\frac{v^2}{\sigma^2}\right) \cdot \alpha
$$

where:
- $\sigma$ controls the width of the Gaussian (typically ~0.5)
- $\alpha$ is the amplitude/dampening factor (typically ~0.3)

This approximation:
- Is smooth and differentiable
- Peaks at threshold (v=0)
- Falls off for voltages far from threshold

In [ ]:
# Import the actual implementation
from v1_jax.nn.spike_functions import gauss_pseudo, spike_gauss

# Parameters used in V1 model
sigma = 0.5
amplitude = 0.3

# Create voltage range
v = jnp.linspace(-2, 2, 200)

# Forward pass: still Heaviside
z = spike_gauss(v, sigma, amplitude)

# Backward pass: Gaussian pseudo-derivative
pseudo = gauss_pseudo(v, sigma, amplitude)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Forward
axes[0].plot(v, z, 'b-', linewidth=2)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Scaled voltage')
axes[0].set_ylabel('Spike output')
axes[0].set_title('Forward: Heaviside (unchanged)')
axes[0].grid(True, alpha=0.3)

# Backward
axes[1].plot(v, pseudo, 'r-', linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Scaled voltage')
axes[1].set_ylabel('Gradient')
axes[1].set_title(f'Backward: Gaussian (σ={sigma}, α={amplitude})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak gradient at v=0: {amplitude:.2f}")
print(f"Gradient at v=±1σ: {float(gauss_pseudo(jnp.array(sigma), sigma, amplitude)):.4f}")
print(f"Gradient at v=±2σ: {float(gauss_pseudo(jnp.array(2*sigma), sigma, amplitude)):.4f}")

### Effect of σ (Width) and α (Amplitude)

Let's explore how the hyperparameters affect the surrogate gradient.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Effect of sigma
for sigma in [0.2, 0.5, 1.0, 2.0]:
    pseudo = gauss_pseudo(v, sigma, amplitude=0.3)
    axes[0].plot(v, pseudo, linewidth=2, label=f'σ={sigma}')

axes[0].set_xlabel('Scaled voltage')
axes[0].set_ylabel('Gradient')
axes[0].set_title('Effect of σ (width)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Effect of amplitude
for amp in [0.1, 0.3, 0.5, 1.0]:
    pseudo = gauss_pseudo(v, sigma=0.5, amplitude=amp)
    axes[1].plot(v, pseudo, linewidth=2, label=f'α={amp}')

axes[1].set_xlabel('Scaled voltage')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Effect of α (amplitude/dampening)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insights:")
print("- Larger σ: Gradients affect neurons further from threshold")
print("- Larger α: Stronger gradients (can lead to exploding gradients)")
print("- Smaller α: More stable but slower learning")

## 3. Implementation with jax.custom_vjp

JAX provides `jax.custom_vjp` (custom Vector-Jacobian Product) to define separate forward and backward passes.

```python
@jax.custom_vjp
def spike_gauss(v_scaled, sigma, amplitude):
    return (v_scaled > 0.0).astype(jnp.float32)  # Forward: Heaviside

def _spike_gauss_fwd(v_scaled, sigma, amplitude):
    z = spike_gauss(v_scaled, sigma, amplitude)
    return z, (v_scaled, sigma, amplitude)  # Return residuals for backward

def _spike_gauss_bwd(res, g):
    v_scaled, sigma, amplitude = res
    grad = g * gauss_pseudo(v_scaled, sigma, amplitude)  # Backward: Gaussian
    return (grad, None, None)  # No gradient for sigma/amplitude

spike_gauss.defvjp(_spike_gauss_fwd, _spike_gauss_bwd)
```

In [ ]:
# Let's verify this works by computing gradients

# Define a simple loss function
def loss_fn(v):
    """Sum of spikes (we want to maximize spiking)."""
    return jnp.sum(spike_gauss(v, sigma=0.5, amplitude=0.3))

# Compute gradient
test_v = jnp.array([-1.0, -0.5, 0.0, 0.5, 1.0])
grads = jax.grad(loss_fn)(test_v)

print("Input voltages:", test_v)
print("Spike outputs: ", spike_gauss(test_v, 0.5, 0.3))
print("Gradients:     ", grads)

# Verify gradient values match Gaussian formula
expected_grads = gauss_pseudo(test_v, sigma=0.5, amplitude=0.3)
print("Expected:      ", expected_grads)
print("Match:", jnp.allclose(grads, expected_grads))

### Why custom_vjp instead of straight-through estimator?

The straight-through estimator (STE) simply uses `grad = 1` everywhere:

```python
# Straight-through estimator (simpler but less accurate)
z = v - jax.lax.stop_gradient(v) + jax.lax.stop_gradient(v > 0)
```

The Gaussian surrogate is preferred because:
1. **Locality**: Gradients are stronger near threshold
2. **Stability**: Bounded gradients prevent explosion
3. **Biological plausibility**: Models uncertainty near threshold

In [ ]:
# Compare STE vs Gaussian surrogate

def spike_ste(v):
    """Straight-through estimator."""
    return v - jax.lax.stop_gradient(v) + jax.lax.stop_gradient((v > 0).astype(jnp.float32))

def loss_ste(v):
    return jnp.sum(spike_ste(v))

def loss_gauss(v):
    return jnp.sum(spike_gauss(v, 0.5, 0.3))

# Compute gradients
v_test = jnp.linspace(-2, 2, 100)
grad_ste = jax.vmap(jax.grad(lambda x: spike_ste(x)))(v_test)
grad_gauss = jax.vmap(jax.grad(lambda x: spike_gauss(x, 0.5, 0.3)))(v_test)

plt.figure(figsize=(10, 4))
plt.plot(v_test, grad_ste, 'b-', linewidth=2, label='STE (constant=1)')
plt.plot(v_test, grad_gauss, 'r-', linewidth=2, label='Gaussian surrogate')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Threshold')
plt.xlabel('Scaled voltage')
plt.ylabel('Gradient')
plt.title('Straight-Through Estimator vs Gaussian Surrogate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("STE: All neurons get equal gradient regardless of distance from threshold")
print("Gaussian: Neurons closer to threshold get stronger gradients")

## 4. Comparing Surrogate Methods

The codebase provides three surrogate gradient implementations:

1. **Gaussian** (`spike_gauss`): $e^{-v^2/\sigma^2} \cdot \alpha$
2. **Piecewise Linear** (`spike_piecewise`): $\alpha \cdot \max(1 - |v|, 0)$
3. **Sigmoid** (`spike_sigmoid`): $\beta \cdot \sigma(\beta v) \cdot (1 - \sigma(\beta v))$

In [ ]:
from v1_jax.nn.spike_functions import (
    spike_gauss, gauss_pseudo,
    spike_piecewise, pseudo_derivative,
    spike_sigmoid
)

v = jnp.linspace(-2, 2, 200)

# Compute gradients for each method
grad_gauss = jax.vmap(jax.grad(lambda x: spike_gauss(x, 0.5, 0.3)))(v)
grad_piecewise = jax.vmap(jax.grad(lambda x: spike_piecewise(x, 0.3)))(v)
grad_sigmoid = jax.vmap(jax.grad(lambda x: spike_sigmoid(x, 10.0)))(v)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Gaussian
axes[0].plot(v, grad_gauss, 'r-', linewidth=2)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Scaled voltage')
axes[0].set_ylabel('Gradient')
axes[0].set_title('Gaussian (σ=0.5, α=0.3)')
axes[0].grid(True, alpha=0.3)

# Piecewise
axes[1].plot(v, grad_piecewise, 'g-', linewidth=2)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Scaled voltage')
axes[1].set_ylabel('Gradient')
axes[1].set_title('Piecewise Linear (α=0.3)')
axes[1].grid(True, alpha=0.3)

# Sigmoid
axes[2].plot(v, grad_sigmoid, 'm-', linewidth=2)
axes[2].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Scaled voltage')
axes[2].set_ylabel('Gradient')
axes[2].set_title('Sigmoid (β=10)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# All methods in one plot
plt.figure(figsize=(10, 5))
plt.plot(v, grad_gauss, 'r-', linewidth=2, label='Gaussian')
plt.plot(v, grad_piecewise, 'g-', linewidth=2, label='Piecewise')
plt.plot(v, grad_sigmoid, 'm-', linewidth=2, label='Sigmoid')
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Threshold')
plt.xlabel('Scaled voltage')
plt.ylabel('Surrogate Gradient')
plt.title('Comparison of Surrogate Gradient Methods')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Method Comparison:")
print("- Gaussian: Smooth, widely used, good default choice")
print("- Piecewise: Computationally simple, hard cutoff at |v|=1")
print("- Sigmoid: Similar to Gaussian, tunable sharpness via β")

## 5. Gradient Flow Verification

Let's verify that gradients flow correctly through a simple SNN-like computation.

In [ ]:
# Simple 2-layer SNN computation
def simple_snn(weights, inputs):
    """Simple 2-layer SNN: input -> spikes -> weighted sum."""
    # Layer 1: weighted sum + spike
    h = inputs @ weights['w1']
    z = spike_gauss(h, sigma=0.5, amplitude=0.3)
    
    # Layer 2: weighted sum (no spike, just linear readout)
    out = z @ weights['w2']
    return out

def loss(weights, inputs, targets):
    """MSE loss."""
    pred = simple_snn(weights, inputs)
    return jnp.mean((pred - targets) ** 2)

# Initialize
key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)

weights = {
    'w1': jax.random.normal(k1, (10, 20)) * 0.1,
    'w2': jax.random.normal(k2, (20, 5)) * 0.1,
}

inputs = jax.random.normal(jax.random.PRNGKey(0), (32, 10))
targets = jax.random.normal(jax.random.PRNGKey(1), (32, 5))

# Compute gradients
grads = jax.grad(loss)(weights, inputs, targets)

print("Gradient shapes:")
print(f"  w1: {grads['w1'].shape}")
print(f"  w2: {grads['w2'].shape}")

print("\nGradient statistics:")
print(f"  w1 mean: {float(jnp.mean(grads['w1'])):.6f}")
print(f"  w1 std:  {float(jnp.std(grads['w1'])):.6f}")
print(f"  w2 mean: {float(jnp.mean(grads['w2'])):.6f}")
print(f"  w2 std:  {float(jnp.std(grads['w2'])):.6f}")

print("\nNon-zero gradients:")
print(f"  w1: {jnp.sum(grads['w1'] != 0)} / {grads['w1'].size}")
print(f"  w2: {jnp.sum(grads['w2'] != 0)} / {grads['w2'].size}")

In [ ]:
# Visualize gradient flow during training
losses = []
grad_norms = {'w1': [], 'w2': []}

lr = 0.01
current_weights = jax.tree.map(lambda x: x.copy(), weights)

for step in range(100):
    l, grads = jax.value_and_grad(loss)(current_weights, inputs, targets)
    losses.append(float(l))
    grad_norms['w1'].append(float(jnp.linalg.norm(grads['w1'])))
    grad_norms['w2'].append(float(jnp.linalg.norm(grads['w2'])))
    
    # SGD update
    current_weights = jax.tree.map(
        lambda w, g: w - lr * g, current_weights, grads
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(losses, 'b-', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(grad_norms['w1'], label='w1', linewidth=2)
axes[1].plot(grad_norms['w2'], label='w2', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norms Over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
print(f"Reduction:    {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

---

## Summary

Key takeaways:

1. **The Problem**: The Heaviside function has zero gradient, preventing backpropagation in SNNs

2. **The Solution**: Surrogate gradients approximate the derivative in the backward pass while keeping the discrete spike in the forward pass

3. **Gaussian Pseudo-Derivative**: $e^{-v^2/\sigma^2} \cdot \alpha$ provides smooth, localized gradients

4. **JAX Implementation**: `jax.custom_vjp` cleanly separates forward (Heaviside) and backward (Gaussian) computations

5. **Hyperparameters**:
   - σ (sigma): Controls gradient width. Default: 0.5
   - α (amplitude): Controls gradient magnitude. Default: 0.3

---

## Exercises

1. **Experiment with σ**: Try σ values of 0.1, 0.5, and 2.0. How does training stability change?

2. **Implement your own surrogate**: Try a triangular pseudo-derivative:
   $$\frac{\partial z}{\partial v} = \max(0, 1 - |v|)$$

3. **Compare convergence**: Train the simple SNN above with Gaussian vs Piecewise surrogate. Which converges faster?

In [ ]:
# Exercise starter: Implement triangular surrogate

@jax.custom_vjp
def spike_triangular(v_scaled, amplitude):
    """Spike with triangular surrogate gradient."""
    return (v_scaled > 0.0).astype(jnp.float32)

def _spike_triangular_fwd(v_scaled, amplitude):
    # TODO: Implement forward pass
    pass

def _spike_triangular_bwd(res, g):
    # TODO: Implement backward pass with triangular gradient
    pass

# spike_triangular.defvjp(_spike_triangular_fwd, _spike_triangular_bwd)

print("Implement the triangular surrogate gradient as an exercise!")

---

## Source Files

- `src/v1_jax/nn/spike_functions.py` - All spike functions with custom gradients

## References

- Chen et al., "Data-driven models of visual cortex", Science Advances 2022
- Neftci et al., "Surrogate Gradient Learning in Spiking Neural Networks", IEEE Signal Processing 2019